In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_openai import OpenAIEmbeddings
from supabase import create_client

ROOT = Path("..").resolve()
ENV_PATH = ROOT / "backend/.env"
PDF_PATH = ROOT / "backend/test_docs/tsla-10k-2024.pdf"

def load_env_file(path: Path):
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if value and key not in os.environ:
            os.environ[key] = value

load_env_file(ENV_PATH)

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

/home/hoangvu/Coursework_Y4/DataFlatform/final-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

for doc in docs:
    doc.metadata["filename"] = PDF_PATH.name

len(docs), docs[0].metadata, docs[0].page_content[:800]

(157,
 {'producer': 'Skia/PDF m146',
  'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0',
  'creationdate': '2026-04-11T19:50:50+00:00',
  'title': 'tsla-20241231',
  'moddate': '2026-04-11T19:50:50+00:00',
  'source': '/home/hoangvu/Coursework_Y4/DataFlatform/final-project/backend/test_docs/tsla-10k-2024.pdf',
  'total_pages': 157,
  'page': 0,
  'page_label': '1',
  'filename': 'tsla-10k-2024.pdf'},
 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\nx ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended December 31, 2024\nOR\no TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from _________ to _________\nCommission File Number: 001-34756\nTesla, Inc.\n(Exact name of registrant as specified in its charter)\nTexas 91-2197729\n

In [3]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

vector_store = SupabaseVectorStore(
    client=supabase,
    embedding=embeddings,
    table_name="documents",
    query_name="match_documents",
)

In [4]:
ids = vector_store.add_documents(docs)
len(ids), ids[:5]

(157,
 ['1db17107-e869-4ebe-9fc3-2ba7919cf953',
  'f40d746f-7067-4364-bb09-48083c8280ff',
  '10646a5e-5ec9-4905-9310-7a777473faa3',
  '67c65e8a-de2b-4422-902b-df03852ced0e',
  '2767cafb-38d7-42e1-b34c-75ad4fc469c0'])

In [5]:
query = "What is Tesla's mission?"

query_embedding = vector_store.embeddings.embed_query(query)
match_documents_params = vector_store.match_args(query_embedding, None)
response = vector_store._client.rpc(
    vector_store.query_name,
    match_documents_params,
).limit(5).execute()

results = [
    {
        "page": item["metadata"].get("page_label") or item["metadata"].get("loc", {}).get("pageNumber"),
        "filename": item["metadata"].get("filename"),
        "preview": item["content"][:300],
    }
    for item in response.data
]

results

[{'page': '21',
  'filename': 'tsla-10k-2024.pdf',
  'preview': 'Table of Contents\nWe believe that sound corporate governance is critical to helping us achieve our goals, including with respect to ESG. We\ncontinue to evolve a governance framework that exercises appropriate oversight of responsibilities at all levels throughout the company\nand manages its affairs '},
 {'page': '22',
  'filename': 'tsla-10k-2024.pdf',
  'preview': 'Table of Contents\n• Engineering Development Program – Launched 2024, the program focuses on developing recent college and university\ngraduates for specialized engineering fields. In partnership with local education partners, the program educates early-career\nengineers in controls engineering, enhanc'},
 {'page': '20',
  'filename': 'tsla-10k-2024.pdf',
  'preview': 'Table of Contents\nWe believe that there is also increasing competition for our vehicle offerings as a platform for delivering self-driving\ntechnologies, charging solutions and other feature